In [3]:
%pip install Faker


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import date, timedelta

# ============================================================
# Recruitment Analytics Project
# Synthetic Dataset Generation
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker()

# ============================================================
# 1. REFERENCE DATA
# ============================================================

SOURCES = [
    "LinkedIn", "linkedin", "LINKEDIN", "Linked in",
    "Indeed", "indeed", "INDEED", "Indeed ",
    "Naukri", "naukri", "NAUKRI",
    "Referral", "referral", "REFERRAL",
    "Company Website", "Company website", "company website",
    "Job Fair", "job fair", "JobFair"
]

JOB_ROLES = [
    "Data Analyst",
    "Data Scientist",
    "Software Engineer",
    "Business Analyst",
    "Product Manager",
    "UX Designer",
    "DevOps Engineer",
    "QA Engineer",
    "HR Executive",
    "Sales Associate",
    "Marketing Specialist"
]

DEPARTMENTS = [
    "Engineering",
    "Engineering ",
    "Marketing",
    "Marketing ",
    "Finance",
    "Finance ",
    "Human Resources",
    "Human Resources ",
    "Sales",
    "Sales ",
    "Product",
    "Product ",
    "Operations",
    "Operations "
]

RECRUITERS = [
    "Priya Sharma",
    "David Lee",
    "Sarah Johnson",
    "Arjun Patel",
    "Rohan Mehta",
    "Emily Chen",
    "Aisha Khan",
    "Vikram Singh"
]

# ============================================================
# 2. DATASET SETTINGS
# ============================================================

NUM_APPLICANTS = 1000

START_DATE = date(2025, 9, 1)
END_DATE = date(2026, 9, 30)

DATE_RANGE = (END_DATE - START_DATE).days

# Cumulative funnel targets:
# Applied      = 1000
# Screened     = approximately 600
# Interviewed  = approximately 300
# Offered      = approximately 100
# Hired        = approximately 80

# ============================================================
# 3. GENERATE APPLICANT RECORDS
# ============================================================

rows = []

for applicant_id in range(1, NUM_APPLICANTS + 1):

    # Applicant information
    name = fake.name()
    age = random.randint(18, 45)

    source = random.choice(SOURCES)
    job_role = random.choice(JOB_ROLES)
    department = random.choice(DEPARTMENTS)
    recruiter = random.choice(RECRUITERS)

    # Random application date
    applied_date = START_DATE + timedelta(
        days=random.randint(0, DATE_RANGE)
    )

    # --------------------------------------------------------
    # Recruitment funnel progression
    # --------------------------------------------------------

    outcome = np.random.random()

    if outcome < 0.08:
        stage = 5       # Hired
    elif outcome < 0.10:
        stage = 4       # Offered, not hired
    elif outcome < 0.30:
        stage = 3       # Interviewed, no offer
    elif outcome < 0.60:
        stage = 2       # Screened, no interview
    else:
        stage = 1       # Applied only

    # --------------------------------------------------------
    # Generate stage dates only when the applicant reaches
    # the corresponding stage
    # --------------------------------------------------------

    screened_date = pd.NaT
    interviewed_date = pd.NaT
    offered_date = pd.NaT
    hired_date = pd.NaT

    if stage >= 2:
        screened_date = applied_date + timedelta(
            days=random.randint(2, 7)
        )

    if stage >= 3:
        interviewed_date = screened_date + timedelta(
            days=random.randint(3, 14)
        )

    if stage >= 4:
        offered_date = interviewed_date + timedelta(
            days=random.randint(2, 8)
        )

    if stage >= 5:
        hired_date = offered_date + timedelta(
            days=random.randint(4, 18)
        )

    # --------------------------------------------------------
    # Store applicant record
    # --------------------------------------------------------

    rows.append({
        "applicant_id": applicant_id,
        "name": name,
        "age": age,
        "source": source,
        "job_role": job_role,
        "department": department,
        "recruiter": recruiter,
        "applied_date": applied_date,
        "screened_date": screened_date,
        "interviewed_date": interviewed_date,
        "offered_date": offered_date,
        "hired_date": hired_date
    })

# Convert records into a Pandas DataFrame
df = pd.DataFrame(rows)

# ============================================================
# 4. INJECT INVALID AGE VALUES
# ============================================================

invalid_age_indices = random.sample(
    list(df.index),
    12
)

invalid_ages = [
    -5, -1, -1, -5,
    100, 120, 150, 999,
    -3, 110, 999, -1
]

for idx, invalid_age in zip(
    invalid_age_indices,
    invalid_ages
):
    df.loc[idx, "age"] = invalid_age

# ============================================================
# 5. INJECT DUPLICATE APPLICANT IDs
# ============================================================

duplicate_indices = random.sample(
    list(df.index),
    10
)

duplicate_rows = df.loc[duplicate_indices].copy()

# Preserve applicant IDs but introduce slight inconsistencies
for idx in duplicate_rows.index:

    if random.random() < 0.7:
        duplicate_rows.loc[idx, "name"] = fake.name()

    if random.random() < 0.5:
        duplicate_rows.loc[idx, "source"] = random.choice(SOURCES)

# Append duplicate records
df = pd.concat(
    [df, duplicate_rows],
    ignore_index=True
)

# ============================================================
# 6. INJECT MISSING VALUES
# ============================================================

# Some applicants may have missing ages
missing_age_indices = random.sample(
    list(df.index),
    5
)

df.loc[missing_age_indices, "age"] = np.nan

# ============================================================
# 7. SHUFFLE RECORDS
# ============================================================

df = df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# ============================================================
# 8. EXPORT THE RAW DATASET
# ============================================================

df.to_csv(
    "applicant_tracking.csv",
    index=False,
    date_format="%Y-%m-%d"
)

# ============================================================
# 9. GENERATION SUMMARY
# ============================================================

print("=" * 55)
print("RECRUITMENT DATASET GENERATED SUCCESSFULLY")
print("=" * 55)

print(f"\nTotal records: {len(df)}")
print(f"Unique applicant IDs: {df['applicant_id'].nunique()}")
print(f"Duplicate applicant IDs: {df['applicant_id'].duplicated().sum()}")

print("\nDataset columns:")
print(df.columns.tolist())

print("\nRecruitment funnel:")
print(
    pd.DataFrame({
        "Stage": [
            "Applied",
            "Screened",
            "Interviewed",
            "Offered",
            "Hired"
        ],
        "Applicants": [
            len(df),
            df["screened_date"].notna().sum(),
            df["interviewed_date"].notna().sum(),
            df["offered_date"].notna().sum(),
            df["hired_date"].notna().sum()
        ]
    }).to_string(index=False)
)

print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst five records:")
print(df.head())

print("\nSaved as: recuitment_funnel_v1.csv")